### Test des tables possibles de pipeline param (test avec class simple)

In [1206]:
from enum import Enum
from typing import Optional
from datetime import datetime
import json

In [ ]:
# Mappage des paramettre de transformation (mook test de BDD)
class ConditionOperator(Enum):
    GT = ">"     
    LT = "<"     
    GTE = ">="    
    LTE = "<="    
    EQ = "="     
    NEQ = "!="    
    IN = "in"
    NOT_IN = "not_in"



class ETLTransformationCondition:
    def __init__(self,
                 id_condition: int,
                 id_transformation: int,
                 groupe_code: int,
                 operator: ConditionOperator,
                 value_num: float = None,
                 value_str: str = None,
                 value_date: datetime = None):
        
        self.id_condition = id_condition
        self.id_transformation = id_transformation
        self.groupe_code = groupe_code  # les conditions sont reliées à une transformation, si elles sont dans le même groupe alors ce sont des AND, sinon c'est d'OR
        self.operator = operator
        self.value_num = value_num
        self.value_str = value_str
        self.value_date = value_date


class ConditionFailBehavior(Enum):
    SKIP = "skip"       # si la contion passe pas, on passe à la transformation suivante
    STOP = "stop"       # si la contion passe pas, on arrête de transformer
    ERROR = "error"     # si condition passe pas la ligne est une anomalie

# 
class TypeTransformation(Enum):
    MULTIPLY = "multiply"   # multiplication -> fournir valeur float
    DIVIDE = "divide"       # Diviser        -> fournir valeur float
    ADD = "add"             # Additionner    -> fournir valeur float
    SUBTRACT = "subtract"   # Soustraire     -> fournir valeur float
    POWER = "power"         # Puissance      -> fournir valeur float
    ROUND = "round"         # Arrondir       -> fournir int
    FLOOR = "floor"         # arrondir à l'entier inférieur
    CEIL = "ceil"           # arrondir à l'entier supérieur
    CLIP_MIN = "clip_min"   # valeur min pour éviter les abus (si X > 110 -> x = 100)   -> fournir valeur float
    CLIP_MAX = "clip_max"   # valeur max pour éviter les abus (si X > 110 -> x = 100)   -> fournir valeur float

    UPPER = "upper"         # MAJUSCULE
    LOWER = "lower"         # minuscule
    REPLACE = "replace"     # replace - > fournir old/new 
    REGEX_REPLACE = "regex_replace"     # Regex ? #TODO
    SUBSTRING = "substring" # -> start/end

    # Pour les type array,c'est des arrays json type : [xxx, xxx, xx, xxx]
    ARRAY_UNIQUE = "array_unique"   # supprime doublons
    ARRAY_SLICE = "array_slice"     # ex [0, 1, 2]

    # Pour type date ou type datetime, mais comment savoir si c'est une date format anglais ou format fr ??
    ADD_DAYS = "add_days"           # -> fournir int
    ADD_MONTH = "add_month"
    ADD_YEAR = "add_year"   #
    EXTRACT_DAY = "extract_day"   # fournir string dans string 1 
    EXTRACT_MONTH = "extract_month" # fournir string dans string 1 et il ressort un nombre comme un 1 pour janvier
    EXTRACT_YEAR = "extract_year"   # fournir string dans string 1
    DEFAULT_DAY = "default_day"     # fournir un entier qui est un numéro de jour
    DEFAULT_MONTH = "default_month" # fournir un entier qui est le numéro du mois
    DEFAULT_YEAR = "default_year"
    ADD_HOUR = "add_hour"
    ADD_MINUTE = "add_minute" 
    ADD_SECOND = "add_second" 
    EXTRACT_HOUR = "extract_hour" 
    EXTRACT_MINUTE = "extract_hour" 
    EXTRACT_SECOND = "extract_hour"


class ETLColumnTransformation:
    def __init__(self,
                 id_transformation: int,
                 id_etl_column_mapping: int,
                 ordre: int,
                 skip_error: bool,
                 type_transformation: TypeTransformation,
                 condition_fail_behavior: ConditionFailBehavior,
                 value_num: float = None,
                 value_int: int = None,
                 value_str: str = None,
                 value_int_2: int = None,
                 value_str_2: str = None,
                 conditions: list = None):
        
        self.id_transformation = id_transformation
        self.id_etl_column_mapping = id_etl_column_mapping
        self.ordre = ordre
        self.skip_error = skip_error
        self.type_transformation = type_transformation
        self.condition_fail_behavior = condition_fail_behavior
        
        self.value_num = value_num
        self.value_int = value_int
        self.value_str = value_str
        self.value_int_2 = value_int_2
        self.value_str_2 = value_str_2

        self.conditions = conditions or []

In [ ]:
class TypeDonnees(Enum):
    INT = "int"
    DECIMAL = "decimal"
    STRING = "string"
    DATE = "date"
    TIMESTAMP = "timestamp"
    BOOLEAN = "boolean"
    ARRAY_DELIMITED_JSON = "array_delimited_json"    # → "item1;item2;item3"
    ARRAY_JSON = "array_json"              # → '["item1","item2"]'

# Classe de base pour les contraintes
class ColumnConstraint:
    def __init__(self, id_constraint: int):
        self.id_constraint = id_constraint

# Contraintes STRING
class StringConstraint(ColumnConstraint):
    def __init__(self, id_constraint: int, min_length: int = 0, max_length: int = None):
        super().__init__(id_constraint)
        self.min_length = min_length
        self.max_length = max_length

# Contraintes NUMERIC
class NumericConstraint(ColumnConstraint):
    def __init__(self, id_constraint: int, nb_min: float = None,
                 nb_max: float = None, nb_decimal: int = None):
        super().__init__(id_constraint)
        self.nb_min = nb_min
        self.nb_max = nb_max
        self.nb_decimal = nb_decimal

# Contraintes DATE
class DateConstraint(ColumnConstraint):
    def __init__(self, id_constraint: int,
                 date_min: datetime = None,
                 date_max: datetime = None):
        super().__init__(id_constraint)
        self.date_min = date_min
        self.date_max = date_max

# ETLColumnMapping (classe principale)
# Ici on compose avec une contrainte (et pas héritage direct)
class ETLColumnMapping:
    def __init__(self,
                 id_etl_column_mapping: int,
                 colonne_bdd: str,
                 colonne_fichier: str,
                 in_file: bool,
                 type_donnees: TypeDonnees,
                 nullable: bool,
                 valeur_defaut: str,
                 unique_constraint: bool,
                 constraint: ColumnConstraint = None,
                 transformations: ETLColumnTransformation = None):
        
        self.id_etl_column_mapping = id_etl_column_mapping
        self.colonne_bdd = colonne_bdd
        self.colonne_fichier = colonne_fichier
        self.in_file = in_file
        self.type_donnees = type_donnees
        self.nullable = nullable
        self.valeur_defaut = valeur_defaut
        self.unique_constraint = unique_constraint
        
        # la contrainte dépend du type
        self.constraint = constraint
        self.transformations = transformations


class ExtensionFichier(Enum):
    CSV = "csv"
    JSON = "json"

class PipelineETL:
    def __init__(self, id_etl_pipeline: int, libelle: str, table_nom: str,
                 dossier_emplacement: str, nom_fichier_fixe: str,
                 nom_fichier_variable: str,
                 extension_fichier: ExtensionFichier,
                 dossier_clean_emplacement: str,
                 active: bool,
                 colonnes: list[ETLColumnMapping] = None):

        self.id_etl_pipeline = id_etl_pipeline
        self.libelle = libelle
        self.table_nom = table_nom
        self.dossier_emplacement = dossier_emplacement # le dossier ou sont les raws
        self.dossier_clean_emplacement = dossier_clean_emplacement
        self.nom_fichier_fixe = nom_fichier_fixe
        self.nom_fichier_variable = nom_fichier_variable
        self.extension_fichier = extension_fichier
        self.active = active
        self.colonnes = colonnes or []

# MOOK BDD
def get_pipelineBDD_TODO():

    # -------------------------
    # AGE
    # -------------------------
    col_age = ETLColumnMapping(
        id_etl_column_mapping=1,
        colonne_bdd="age",
        colonne_fichier="Age",
        in_file=True,
        type_donnees=TypeDonnees.INT,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=NumericConstraint(1, nb_min=0, nb_max=110, nb_decimal=0),
        transformations=[
            ETLColumnTransformation(
                id_transformation=1,
                id_etl_column_mapping=1,
                ordre=1,
                skip_error=False,
                type_transformation=TypeTransformation.CLIP_MAX,
                condition_fail_behavior=ConditionFailBehavior.ERROR,
                value_num=40,
                conditions=[
                    ETLTransformationCondition(1, 1, 1, ConditionOperator.LT, value_num=50)
                ]
            ),
        ]
    )

    # -------------------------
    # GENDER
    # -------------------------
    col_gender = ETLColumnMapping(
        id_etl_column_mapping=2,
        colonne_bdd="sexe",
        colonne_fichier="Gender",
        in_file=True,
        type_donnees=TypeDonnees.STRING,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=1, max_length=20),
        transformations=[
            # Normalisation en majuscule
            ETLColumnTransformation(
                id_transformation=3,
                id_etl_column_mapping=2,
                ordre=1,
                skip_error=False,
                type_transformation=TypeTransformation.UPPER,
                condition_fail_behavior=ConditionFailBehavior.SKIP
            ),

            # Remplacer valeurs incohérentes
            ETLColumnTransformation(
                id_transformation=4,
                id_etl_column_mapping=2,
                ordre=2,
                skip_error=False,
                type_transformation=TypeTransformation.REPLACE,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_str="FEMALE",
                value_str_2="F",
                conditions=[
                ]
            ),
            ETLColumnTransformation(
                id_transformation=4,
                id_etl_column_mapping=2,
                ordre=2,
                skip_error=False,
                type_transformation=TypeTransformation.SUBSTRING,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_int=1,
                value_int_2=3,
                conditions=[
                ]
            )
        ]
    )

    # -------------------------
    # TYPE MALADIE
    # -------------------------
    col_type_maladie = ETLColumnMapping(
        id_etl_column_mapping=3,
        colonne_bdd="type_maladie",
        colonne_fichier="Disease_Type",
        in_file=True,
        type_donnees=TypeDonnees.STRING,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=2, max_length=50),
        transformations=[
            # Nettoyage string
            ETLColumnTransformation(
                id_transformation=5,
                id_etl_column_mapping=3,
                ordre=1,
                skip_error=False,
                type_transformation=TypeTransformation.LOWER,
                condition_fail_behavior=ConditionFailBehavior.SKIP
            )
        ]
    )

    # -------------------------
    # col_json_array
    # -------------------------
    col_json_array = ETLColumnMapping(
        id_etl_column_mapping=4,
        colonne_bdd="secondary_muscles",
        colonne_fichier="secondaryMuscles",
        in_file=True,
        type_donnees=TypeDonnees.ARRAY_DELIMITED_JSON,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        transformations=[
            ETLColumnTransformation(
                id_transformation=5,
                id_etl_column_mapping=3,
                ordre=1,
                skip_error=False,
                type_transformation=TypeTransformation.ARRAY_UNIQUE,
                condition_fail_behavior=ConditionFailBehavior.SKIP
            ),
            ETLColumnTransformation(
                id_transformation=5,
                id_etl_column_mapping=3,
                ordre=2,
                skip_error=False,
                type_transformation=TypeTransformation.ARRAY_SLICE,
                value_int=0,
                value_int_2=3,
                condition_fail_behavior=ConditionFailBehavior.SKIP
            )
        ]
    )


    # -------------------------
    # test_date
    # -------------------------
    col_date = ETLColumnMapping(
        id_etl_column_mapping=5,
        colonne_bdd="test_date",
        colonne_fichier="date_test",
        in_file=True,
        type_donnees=TypeDonnees.DATE,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=DateConstraint(1, date_max=10/20/2026, date_min="01/01/2022"),
        transformations=[
            ETLColumnTransformation(
                id_transformation=9,
                id_etl_column_mapping=5,
                ordre=1,
                skip_error=False,
                type_transformation=TypeTransformation.ADD_DAYS,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_int=1
            ),
            ETLColumnTransformation(
                id_transformation=9,
                id_etl_column_mapping=5,
                ordre=2,
                skip_error=False,
                type_transformation=TypeTransformation.ADD_MONTH,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_int=1
            ),
            ETLColumnTransformation(
                id_transformation=9,
                id_etl_column_mapping=5,
                ordre=3,
                skip_error=False,
                type_transformation=TypeTransformation.ADD_YEAR,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_int=-1
            ),
            ETLColumnTransformation(
                id_transformation=9,
                id_etl_column_mapping=5,
                ordre=3,
                skip_error=False,
                type_transformation=TypeTransformation.DEFAULT_DAY,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_int=1
            ),
            ETLColumnTransformation(
                id_transformation=9,
                id_etl_column_mapping=5,
                ordre=3,
                skip_error=False,
                type_transformation=TypeTransformation.ADD_HOUR,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_int=9
            ),
            ETLColumnTransformation(
                id_transformation=9,
                id_etl_column_mapping=5,
                ordre=3,
                skip_error=False,
                type_transformation=TypeTransformation.ADD_MINUTE,
                condition_fail_behavior=ConditionFailBehavior.SKIP,
                value_int=30
            ),
        ]
    )

    # -------------------------
    # PIPELINE
    # -------------------------
    pipeline1 = PipelineETL(
        1,
        "Import diet recommendation",
        "gym_members_exercise_tracking_synthetic_data",
        "\\raw\\",
        "gym_members_exercise_tracking_synthetic_data",
        "",
        ExtensionFichier.CSV,
        "/clean",
        True,
        [col_age, col_gender]
    )

    return pipeline1

# Le but de ce notebook

L’objectif ici est de créer des fonctions réutilisables qui seront exploitées dans des pipelines de traitement de données.

Ces fonctions couvrent plusieurs étapes :

- Ouverture et lecture de fichiers (CSV / JSON)
- Nettoyage et vérification des données
- Séparation des lignes valides des anomalies

Ces fonctions doivent être le plus génériques possible, sans dépendre de valeurs codées en dur, afin d’être facilement réutilisables.

J’expérimente l’idée d’un pipeline "écrit en BDD", en schématisant les données plutôt qu’en utilisant de réelles bases de données, pour tester le concept.

## Partie 1 : Ouverture de fichiers

Les fonctions de cette partie servent à ouvrir un ou plusieurs fichiers CSV ou JSON dans le cadre d’une pipeline. Elles permettent :

- De localiser les fichiers dans les dossiers
- De détecter leur type (CSV / JSON)
- De les convertir en DataFrame avec pandas, pour les exploiter ensuite sans se soucier des extensions

In [1209]:
import os
import re
import pandas as pd
import numpy as np

DATA_ROOT = "../data"

### 1.1 Normalisation du chemin

Cette fonction permet de déterminer le dossier où se trouvent les fichiers à traiter.

- Elle se base sur une variable d’environnement comme DATA_ROOT
- On peut ensuite fournir en argument des sous-dossiers où l’on imagine stocker les fichiers CSV ou JSON

Cela garantit que les pipelines restent flexibles et portables.

In [1210]:
# 1. Normalisation du chemin
def normalize_path(dossier_emplacement, data_root=DATA_ROOT):
    """
    Normalise le chemin du dossier relatif à ./data/
    """
    # Normaliser slashes
    dossier_emplacement = dossier_emplacement.replace("\\", "/").replace("//", "/")
    # Supprimer slash en début ou fin
    dossier_emplacement = dossier_emplacement.strip("/")
    # Construire chemin complet
    full_path = os.path.join(data_root, dossier_emplacement)
    # Normaliser le chemin final
    return os.path.normpath(full_path)

### 1.2 Création du motif Regex pour trouver le fichier

Cette fonction génère un motif regex pour identifier les fichiers à traiter.

- Arguments : nom_fixe, nom_variable, extension
    - nom_fixe : première partie du nom du fichier (ex. "Exercice")
    - nom_variable : partie variable du nom du fichier (ex. "YYYYMMDD")
    - extension : type de fichier (csv ou json)

Le motif généré permet de retrouver tous les fichiers correspondants, par exemple :

- CSV commençant par "Exercice"
- Avec 8 caractères pour la date
- Se terminant par .csv

In [1211]:
# 2. Création du motif regex pour le fichier
def build_filename_pattern(nom_fixe, nom_variable, extension):
    if not extension.startswith("."):
        extension = "." + extension
    
    if nom_variable:
        # Le nom variable a une longueur connue : on remplace par autant de "."
        motif = f"^{re.escape(nom_fixe)}{'.' * len(nom_variable)}{re.escape(extension)}$"
    else:
        motif = f"^{re.escape(nom_fixe)}{re.escape(extension)}$"
    
    return motif

### 1.3 Recherche des fichiers correspondants

Cette fonction cherche les fichiers dans un dossier en fonction de la regex fournie :

- Arguments : dossier et motif regex
- Retour : liste des chemins de fichiers correspondants

⚠️ Actuellement, un print est utilisé si le dossier n’existe pas.

- En production, il serait préférable de remonter une erreur ou de logger l’information.

In [1212]:
# 3. Recherche des fichiers correspondants et retourne les path
def find_matching_files(dossier, motif_regex):
    matched_files = []
    if os.path.exists(dossier) and os.path.isdir(dossier):
        for f in os.listdir(dossier):
            if re.match(motif_regex, f):
                matched_files.append(os.path.join(dossier, f))  # chemin complet
    else:
        print(f"[WARNING] Le dossier '{dossier}' n'existe pas.")
    return matched_files


# --- TESTS ---
dossier_emplacement = "\\raw\\"
nom_fichier_fixe = "exercisedb_hobby"
nom_fichier_variable = ""  # par exemple une date
extension_fichier = ".json"

# Normalisation du chemin
normalized_folder = normalize_path(dossier_emplacement)
print("Chemin normalisé :", normalized_folder)

# Construction du motif regex
pattern = build_filename_pattern(nom_fichier_fixe, nom_fichier_variable, extension_fichier)
print("Motif regex :", pattern)

# Recherche des fichiers
matched = find_matching_files(normalized_folder, pattern)
print("Fichiers trouvés :", matched)


# --- Autre exemple ---
dossier_emplacement2 = "raw/"
nom_fichier_fixe2 = "exercisedb_hobby"
nom_fichier_variable2 = ""
extension_fichier2 = "json"

normalized_folder2 = normalize_path(dossier_emplacement2)
pattern2 = build_filename_pattern(nom_fichier_fixe2, nom_fichier_variable2, extension_fichier2)
matched2 = find_matching_files(normalized_folder2, pattern2)

print("\nChemin normalisé 2 :", normalized_folder2)
print("Motif regex 2 :", pattern2)
print("Fichiers trouvés 2 :", matched2)

Chemin normalisé : ..\data\raw
Motif regex : ^exercisedb_hobby\.json$
Fichiers trouvés : ['..\\data\\raw\\exercisedb_hobby.json']

Chemin normalisé 2 : ..\data\raw
Motif regex 2 : ^exercisedb_hobby\.json$
Fichiers trouvés 2 : ['..\\data\\raw\\exercisedb_hobby.json']


### 1.4 Ouverture et lecture des fichiers

Cette fonction prend un **chemin de fichier unique** et :

- Tente de le lire avec pandas selon son type (CSV ou JSON)
- Retourne un **DataFrame unique**
- Pour les CSV :
  - On considère que chaque fichier a une ligne header
  - Les lignes qui ne correspondent pas au nombre de colonnes du header sont automatiquement ignorées

⚠️ Actuellement, un print s’affiche si :

- Ce n’est ni un CSV ni un JSON
- Le fichier n’existe pas ou ne peut pas être lu
- En production, ces prints devraient être remplacés par des logs ou exceptions

💡 Notes complémentaires :

- Une seconde fonction `read_files` permet de traiter une **liste de fichiers**, mais elle utilise simplement `read_single_file` en boucle. Elle est moins utile si l’on souhaite suivre individuellement chaque fichier (logs, comptage des lignes ignorées, etc.)
- On pourrait également comptabiliser le nombre de lignes supprimées dans les CSV pour un suivi et un logging plus précis

In [1213]:
# Lecture automatique avec pandas
def read_single_file_with_pandas(file_path):
    """
    Lit un fichier CSV ou JSON et retourne un DataFrame.
    Retourne None si l'extension n'est pas supportée ou en cas d'erreur.
    """
    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == ".csv":
            df = pd.read_csv(file_path, header=0, on_bad_lines='skip') # header obligatoire et saute les lignes incorrectes (nb colonne different du header)
        elif ext == ".json":
            df = pd.read_json(file_path)
        else:
            print(f"[WARNING] Extension non supportée pour {file_path}, ignoré.")
            return None
        return df
    except Exception as e:
        print(f"[ERROR] Impossible de lire {file_path}: {e}")
        return None


def read_files_with_pandas(file_paths):
    """
    Lit une liste de fichiers CSV ou JSON et retourne une liste de DataFrames
    en utilisant read_single_file_with_pandas.
    """
    dfs = []
    for fpath in file_paths:
        df = read_single_file_with_pandas(fpath)
        if df is not None:
            dfs.append(df)
    return dfs


# ---TEST---
dfs = read_files_with_pandas(matched)
print(f"Nombre de DataFrames lus : {len(dfs)}")
if dfs:
    print(dfs[0].head())  # afficher les 5 premières lignes du premier fichier

Nombre de DataFrames lus : 1
             exerciseId                                   name  \
0  exr_41n2ha5iPFpN3hEJ                                   es     
1  exr_41n2haAabPyN5t8y                                     10   
2  exr_41n2hadPLLFRGvFk        Sliding Floor Pulldown on Towel   
3  exr_41n2hadQgEEX8wDN                            Triceps Dip   
4  exr_41n2haNJ3NA8yCE2  Dumbbell Incline One Arm Hammer Press   

                                            imageUrl              bodyParts  \
0                                               True                [WAIST]   
1  https://cdn.exercisedb.dev/media/w/images/RB7N...   [QUADRICEPS, THIGHS]   
2  https://cdn.exercisedb.dev/media/w/images/9fW9...                 [BACK]   
3  https://cdn.exercisedb.dev/media/w/images/MruS...  [TRICEPS, UPPER ARMS]   
4  https://cdn.exercisedb.dev/media/w/images/qkXo...           [UPPER ARMS]   

                                          equipments exerciseType  \
0                             

### 1.5 Ptt équivalent au downloader ?

In [1214]:
def get_df_matched_files(pipeline: PipelineETL):
    """
    À partir d'une instance PipelineETL, retourne la liste des fichiers/dataframe correspondants.
    """
    # 1. Normalisation du chemin
    normalized_folder = normalize_path(pipeline.dossier_emplacement)
    # print("[debug] get_df_matched_files - normalized_folder : " + normalized_folder)

    # 2. Construction du motif regex
    pattern = build_filename_pattern(
        pipeline.nom_fichier_fixe,
        pipeline.nom_fichier_variable,
        pipeline.extension_fichier.value  # Enum → string
    )
    # print("[debug] get_df_matched_files - pattern : " + pattern)

    # 3. Recherche des fichiers
    matched_files_path = find_matching_files(normalized_folder, pattern)
    # print("[debug] get_df_matched_files - Fichiers trouvés :", matched_files_path)

    # 4. Ouvre les fichier en retournant une liste de dataframe
    df_matched = read_files_with_pandas(matched_files_path)

    return df_matched

## Partie 2 : Nettoyage et vérification des données

### 2.1 ColumnMapper

Responsabilité :

- renommer colonnes
- créer colonnes manquantes

Avec en argument un dataframe et une liste de la class [ETLColumnMapping]

In [1215]:
def column_mapper(
    df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> pd.DataFrame:
    result = pd.DataFrame()

    for m in mappings:
        if m.in_file and m.colonne_fichier in df.columns:
            result[m.colonne_bdd] = df[m.colonne_fichier]
        else:
            result[m.colonne_bdd] = None

    # Convertir toutes les colonnes en string
    result = result.astype(str)

    return result

### 2.2 Créer un dataframe pour mettre les anomalies

In [1216]:
def generate_anomaly_dataframe(mappings: list[ETLColumnMapping]) -> pd.DataFrame:
    columns = [m.colonne_bdd for m in mappings]
    columns += ["erreur"]
    
    result = pd.DataFrame(columns=columns)
    
    return result

### 2.3 Nettoyer les colonnes texte d'un DataFrame (supprime les espaces inutiles)

In [1217]:
def clean_txt(df: pd.DataFrame) -> pd.DataFrame:
    """
    Nettoie les colonnes texte d'un DataFrame :
    - Supprime les espaces en début et fin de chaque cellule (pour les strings uniquement).
    - Remplace les cellules vides, None, "None", "NULL", "null", ou ne contenant que des espaces par NaN.
    - Ne traite que les colonnes contenant des strings (type object), ignore les autres types (listes, dicts, etc.).
    """
    # Travailler sur une copie pour ne pas modifier l'original
    df_clean = df.copy()
    
    # Colonnes texte (type object)
    object_columns = df_clean.select_dtypes(include='object').columns
    
    # Valeurs considérées comme manquantes
    missing_values = {"", " ", "none", "null", "nan", "None", "NULL", "na"}

    for col in object_columns:
        s = df_clean[col]

        s = s.astype(str).str.strip()

        s = s.replace(
            to_replace=r"^\s*$",
            value=np.nan,
            regex=True
        )

        s = s.mask(s.str.lower().isin(missing_values), np.nan)

        df_clean[col] = s

    return df_clean

### 2.4 Transformer les données

##### Évalue une condition simple sur une colonne, retourne un masque booléen

In [1218]:
def evaluate_single_condition(series: pd.Series, condition: ETLTransformationCondition) -> pd.Series:
    """
    Évalue une condition simple sur une colonne, retourne un masque booléen. UTILISE UNE SERIES DEJA CONVERTIE
    """
    if condition.value_num is not None:
        val = condition.value_num
    elif condition.value_str is not None:
        val = condition.value_str
    elif condition.value_date is not None:
        val = condition.value_date  # peut être None si rien n’est défini
    else:
        print("aucune value_num, value_str, value_date")
        return pd.Series(False, index=series.index)

    # Compare chaque ligne selon l'opérateur de la condition
    if condition.operator == ConditionOperator.GT:
        return series > val

    elif condition.operator == ConditionOperator.LT:
        return series < val

    elif condition.operator == ConditionOperator.GTE:
        return series >= val

    elif condition.operator == ConditionOperator.LTE:
        return series <= val

    elif condition.operator == ConditionOperator.EQ:
        return series == val

    elif condition.operator == ConditionOperator.NEQ:
        return series != val

    elif condition.operator == ConditionOperator.IN:
        return series.isin(val)

    elif condition.operator == ConditionOperator.NOT_IN:
        return ~series.isin(val)

    # Par défaut : aucune ligne ne passe
    return pd.Series(False, index=series.index)


##### Évalue un ensemble de conditions pour une colonne

In [1219]:
def evaluate_conditions(series: pd.Series, conditions: list[ETLTransformationCondition]) -> pd.Series:
    """
    Évalue un ensemble de conditions pour une colonne.
    Retourne un masque booléen indiquant quelles lignes passent les conditions.
    """
    if not conditions:
        return pd.Series(True, index=series.index)

    # On regroupe les conditions par groupe_code pour appliquer une logique AND à l'intérieur du groupe
    groups = {}
    for cond in conditions:
        groups.setdefault(cond.groupe_code, []).append(cond)

    group_results = []

    for group in groups.values():
        group_mask = pd.Series(True, index=series.index)

        for condition in group:
            group_mask &= evaluate_single_condition(series, condition) # AND entre conditions du groupe

        group_results.append(group_mask)

    # OR entre les groupes pour respecter la logique "au moins un groupe est vrai"
    final_mask = group_results[0]
    for gm in group_results[1:]:
        final_mask |= gm

    return final_mask


##### Ajoute les lignes ayant échoué ou créant anomalies à la table anomalies

In [1220]:
def add_to_anomalies(df, df_original, anomalies, mask, message):
    """
    Ajoute les lignes ayant échoué ou créant anomalies à la table anomalies.
    """
    if not mask.any():
        return anomalies

    # Copie des lignes originales pour garder les valeurs initiales
    rows = df_original.loc[mask].copy()
    rows["erreur"] = message

    return pd.concat([anomalies, rows], ignore_index=True)


##### Gestion des lignes qui ne passent pas les conditions (SKIP, STOP, ERROR)

In [1221]:
def handle_condition_failure(
    df,
    df_original,
    anomalies,
    transformation,
    col,
    active_mask,
    error_mask_global,
    fail_mask
):
    """
    Gestion des lignes qui ne passent pas les conditions.
    SKIP : ignore les lignes
    STOP : retire les lignes de l'application future
    ERROR : ajoute aux anomalies
    """
    if not fail_mask.any():
        return active_mask, error_mask_global, anomalies

    if transformation.condition_fail_behavior == ConditionFailBehavior.SKIP:
        # On ne fait rien, la transformation sera appliquée seulement aux lignes valides par le parent
        return active_mask, error_mask_global, anomalies

    elif transformation.condition_fail_behavior == ConditionFailBehavior.STOP:
        # Les lignes qui échouent sont retirées du flux actif (stop la transformation pour cette colonne)
        active_mask = active_mask & ~fail_mask
        return active_mask, error_mask_global, anomalies

    elif transformation.condition_fail_behavior == ConditionFailBehavior.ERROR:
        # Les lignes en échec deviennent anomalies
        anomalies = add_to_anomalies(
            df,
            df_original,
            anomalies,
            fail_mask,
            f"Condition non respectée col {col} transfo {transformation.ordre}"
        )

        active_mask = active_mask & ~fail_mask
        error_mask_global = error_mask_global | fail_mask

        return active_mask, error_mask_global, anomalies

    return active_mask, error_mask_global, anomalies


##### Applique la transformation définie sur les lignes sélectionnées

In [1222]:
def apply_transformation(series: pd.Series, transformation, mask):
    """
    Applique la transformation définie sur les lignes sélectionnées (dejà dans le type utile).
    Chaque type de transformation (MULTIPLY, DIVIDE, ADD, etc.) est géré avec pandas.
    """
    try:
        result = series.copy()
        result = series.copy().astype(object)

        # type numérique
        if transformation.type_transformation == TypeTransformation.MULTIPLY:
            result[mask] = series[mask] * transformation.value_num

        elif transformation.type_transformation == TypeTransformation.DIVIDE:
            result[mask] = series[mask] / transformation.value_num

        elif transformation.type_transformation == TypeTransformation.ADD:
            result[mask] = series[mask] + transformation.value_num

        elif transformation.type_transformation == TypeTransformation.SUBTRACT:
            result[mask] = series[mask] - transformation.value_num

        elif transformation.type_transformation == TypeTransformation.ROUND:
            result[mask] = series[mask].round(transformation.value_int)

        elif transformation.type_transformation == TypeTransformation.CLIP_MAX:
            result[mask] = series[mask].clip(upper=transformation.value_num)

        elif transformation.type_transformation == TypeTransformation.CLIP_MIN:
            result[mask] = series[mask].clip(lower=transformation.value_num)

        elif transformation.type_transformation == TypeTransformation.POWER:
            result[mask] = series[mask] ** transformation.value_num
            
        elif transformation.type_transformation == TypeTransformation.FLOOR:
            # Arrondi à l'entier inférieur
            result[mask] = np.floor(series[mask])

        elif transformation.type_transformation == TypeTransformation.CEIL:
            # Arrondi à l'entier supérieur
            result[mask] = np.ceil(series[mask])

        
        # Type string
        elif transformation.type_transformation == TypeTransformation.UPPER:
            result[mask] = series[mask].astype(str).str.upper()

        elif transformation.type_transformation == TypeTransformation.LOWER:
            result[mask] = series[mask].astype(str).str.lower()

        elif transformation.type_transformation == TypeTransformation.REPLACE:
            result[mask] = series[mask].replace(
                transformation.value_str,
                transformation.value_str_2
            )

        elif transformation.type_transformation == TypeTransformation.REGEX_REPLACE:
            result[mask] = series[mask].astype(str).str.replace(
                transformation.value_str,
                transformation.value_str_2,
                regex=True
            )

        elif transformation.type_transformation == TypeTransformation.SUBSTRING:
            result[mask] = series[mask].astype(str).str.slice(
                transformation.value_int,
                transformation.value_int_2
            )

        
        # Type datetime
        elif transformation.type_transformation == TypeTransformation.ADD_DAYS:
            # Ajouter un nombre de jours (int) à la date
            result[mask] = series[mask] + pd.to_timedelta(transformation.value_int, unit='d')

        elif transformation.type_transformation == TypeTransformation.ADD_MONTH:
            # Ajouter un nombre de mois (int) à la date
            result[mask] = series[mask] + pd.DateOffset(months=transformation.value_int)

        elif transformation.type_transformation == TypeTransformation.ADD_YEAR:
            # Ajouter un nombre d'années (int) à la date
            result[mask] = series[mask] + pd.DateOffset(years=transformation.value_int)

        elif transformation.type_transformation == TypeTransformation.EXTRACT_DAY:
            # Extraire le jour
            result[mask] = series[mask].dt.day

        elif transformation.type_transformation == TypeTransformation.EXTRACT_MONTH:
            # Extraire le mois
            result[mask] = series[mask].dt.month

        elif transformation.type_transformation == TypeTransformation.EXTRACT_YEAR:
            # Extraire l'année
            result[mask] = series[mask].dt.year

        elif transformation.type_transformation == TypeTransformation.DEFAULT_DAY:
            # Remplacer le jour par une valeur par défaut (int)
            result[mask] = series[mask].apply(lambda x: x.replace(day=transformation.value_int))

        elif transformation.type_transformation == TypeTransformation.DEFAULT_MONTH:
            # Remplacer le mois par une valeur par défaut (int)
            result[mask] = series[mask].apply(lambda x: x.replace(month=transformation.value_int))

        elif transformation.type_transformation == TypeTransformation.DEFAULT_YEAR:
            # Remplacer l'année par une valeur par défaut (int)
            result[mask] = series[mask].apply(lambda x: x.replace(year=transformation.value_int))

        # --- Ajout d'heures, minutes, secondes ---
        elif transformation.type_transformation == TypeTransformation.ADD_HOUR:
            result[mask] = series[mask] + pd.to_timedelta(transformation.value_int, unit='h')

        elif transformation.type_transformation == TypeTransformation.ADD_MINUTE:
            result[mask] = series[mask] + pd.to_timedelta(transformation.value_int, unit='m')

        elif transformation.type_transformation == TypeTransformation.ADD_SECOND:
            result[mask] = series[mask] + pd.to_timedelta(transformation.value_int, unit='s')

        # --- Extraction d'heure, minute, seconde ---
        elif transformation.type_transformation == TypeTransformation.EXTRACT_HOUR:
            result[mask] = series[mask].dt.hour

        elif transformation.type_transformation == TypeTransformation.EXTRACT_MINUTE:
            result[mask] = series[mask].dt.minute

        elif transformation.type_transformation == TypeTransformation.EXTRACT_SECOND:
            result[mask] = series[mask].dt.second


        # Type array
        elif transformation.type_transformation == TypeTransformation.ARRAY_UNIQUE:
            result[mask] = series[mask].apply(
                lambda x: list(dict.fromkeys(x)) if isinstance(x, list) else x
            )

        elif transformation.type_transformation == TypeTransformation.ARRAY_SLICE:
            result[mask] = series[mask].apply(
                lambda x: x[
                    transformation.value_int : transformation.value_int_2
                ] if isinstance(x, list) else x
            )

        return result

    except Exception as e:
        print(f"DEBUG, ici on est de le catch, la transformation type était : {transformation.type_transformation}")
        print(f"series.dtype = {series.dtype}")
        display(series)
        #print(f"transformation.value_int = {transformation.value_int}")
        print(f"Erreur exacte : {e}")
        series[mask] = np.nan
        return series


##### Cherche le type d'une colonnes en fonction de sa transformation

In [1223]:
def get_expected_type(transformation: ETLColumnTransformation):
    t = transformation.type_transformation

    if t in {
        TypeTransformation.MULTIPLY,
        TypeTransformation.DIVIDE,
        TypeTransformation.ADD,
        TypeTransformation.SUBTRACT,
        TypeTransformation.ROUND,
        TypeTransformation.CLIP_MAX,
        TypeTransformation.CLIP_MIN,
        TypeTransformation.POWER,
        TypeTransformation.CEIL,
        TypeTransformation.FLOOR,
    }:
        return "numeric"

    elif t in {
        TypeTransformation.UPPER,
        TypeTransformation.LOWER,
        TypeTransformation.REPLACE,
        TypeTransformation.REGEX_REPLACE,
        TypeTransformation.SUBSTRING,
    }:
        return "string"

    elif t in {
        TypeTransformation.ADD_DAYS,
        TypeTransformation.ADD_MONTH,
        TypeTransformation.ADD_YEAR,
        TypeTransformation.EXTRACT_DAY,
        TypeTransformation.EXTRACT_MONTH,
        TypeTransformation.ADD_HOUR,
        TypeTransformation.ADD_MINUTE,
        TypeTransformation.ADD_SECOND,
        TypeTransformation.EXTRACT_HOUR,
        TypeTransformation.EXTRACT_MINUTE,
        TypeTransformation.EXTRACT_SECOND,
        TypeTransformation.DEFAULT_DAY,
        TypeTransformation.DEFAULT_MONTH,
        TypeTransformation.DEFAULT_YEAR,
    }:
        return "datetime"
    elif t in {
        TypeTransformation.ARRAY_UNIQUE,
        TypeTransformation.ARRAY_SLICE,
    }:
        return "array"

    return "string"

##### Convertie une colonnes dans un type

In [1224]:
import ast

def safe_parse_array(val):
    if pd.isna(val):
        return np.nan

    if isinstance(val, list):
        return val

    try:
        # Cas JSON propre
        return json.loads(val)
    except:
        try:
            # Cas Python style ['a','b']
            return ast.literal_eval(val)
        except:
            return np.nan

def convert_series(series: pd.Series, target_type: str) -> pd.Series:
    if target_type == "numeric":
        return pd.to_numeric(series, errors="coerce").astype(float)

    elif target_type == "datetime":
        return pd.to_datetime(series, errors="coerce")
    
    elif target_type == "array":
        return series.apply(safe_parse_array)

    elif target_type == "string":
        return series.astype(str)

    return series

##### Fonction principale pour appliquer des transformations sur un DataFrame

In [1225]:
def apply_transformations(df, anomalies, mappings):
    """
    Applique toutes les transformations définies dans les mappings sur un DataFrame.

    Pipeline par transformation :
    1. Clean string
    2. Conversion type cible (coerce → NaN)
    3. Détection erreurs de conversion → anomalies
    4. Conditions
    5. Transformation
    6. Retour en string
    7. Clean
    8. Détection erreurs post-transfo → anomalies
    """

    # Copie de travail
    df_clean = df.copy()

    # Copie originale pour log anomalies (important : jamais modifiée)
    df_original = df.copy()

    # Masque global des lignes déjà en erreur (exclues du pipeline)
    error_mask_global = pd.Series(False, index=df_clean.index)

    # BOUCLE SUR LES COLONNES
    for mapping in mappings:
        col = mapping.colonne_bdd

        if col not in df_clean.columns:
            continue
        if not mapping.transformations:
            continue

        # Lignes actives = non nulles + pas déjà en erreur
        active_mask = ~df_clean[col].isna() & ~error_mask_global

        # Tri des transformations
        transformations_sorted = sorted(mapping.transformations, key=lambda t: t.ordre)

        # BOUCLE SUR LES TRANSFORMATIONS
        for transformation in transformations_sorted:

            working_mask = active_mask.copy()

            if not working_mask.any():
                break

            # 1. CLEAN STRING
            df_clean[[col]] = clean_txt(df_clean[[col]])

            # Met à jour le masque après nettoyage
            working_mask = working_mask & ~df_clean[col].isna()

            if not working_mask.any():
                break

            # 2. CONVERSION TYPE CIBLE
            target_type = get_expected_type(transformation)

            # Conversion SAFE (erreurs → NaN)
            converted = convert_series(df_clean[col], target_type)

            # 3. DETECTION ERREURS CONVERSION
            # On détecte uniquement les nouvelles erreurs (pas les NaN d'origine)
            conversion_error_mask = (
                working_mask
                & converted.isna()
                & ~df_clean[col].isna()
            )

            if conversion_error_mask.any():
                anomalies = add_to_anomalies(
                    df_clean,
                    df_original,
                    anomalies,
                    conversion_error_mask,
                    f"Erreur conversion type col {col} transfo {transformation.ordre}"
                )

                # On exclut ces lignes du pipeline
                active_mask = active_mask & ~conversion_error_mask
                error_mask_global |= conversion_error_mask

            # Mise à jour du working_mask après exclusion
            working_mask = active_mask.copy()

            if not working_mask.any():
                break

            # 4. EVALUATION CONDITIONS
            # Conditions évaluées sur la version typée
            condition_mask = evaluate_conditions(converted, transformation.conditions)

            # Sécurité : NaN dans condition → False
            condition_mask = condition_mask.fillna(False)

            condition_mask &= working_mask

            # Lignes qui échouent les conditions
            fail_mask = working_mask & ~condition_mask

            # Gestion comportement (SKIP / STOP / ERROR)
            active_mask, error_mask_global, anomalies = handle_condition_failure(
                df_clean,
                df_original,
                anomalies,
                transformation,
                col,
                active_mask,
                error_mask_global,
                fail_mask
            )

            # Recalcul working_mask après gestion
            working_mask = active_mask.copy()

            if not working_mask.any():
                break

            # 5. LIGNES A TRANSFORMER
            success_mask = working_mask & condition_mask

            if not success_mask.any():
                continue

            # Sauvegarde AVANT transformation
            previous_values = df_clean[col].copy()

            # 6. TRANSFORMATION
            transformed = apply_transformation(converted, transformation, success_mask)

            # 7. RETOUR EN STRING
            df_clean.loc[success_mask, col] = (
                transformed[success_mask]
                .astype(str)
                .replace("nan", np.nan)  # évite string "nan"
            )

            # 8. CLEAN POST-TRANSFORMATION
            df_clean[[col]] = clean_txt(df_clean[[col]])

            # 9. DETECTION ERREURS POST-TRANSFO
            new_error_mask = (
                success_mask
                & df_clean[col].isna()
                & ~previous_values.isna()
            )

            if new_error_mask.any():
                anomalies = add_to_anomalies(
                    df_clean,
                    df_original,
                    anomalies,
                    new_error_mask,
                    f"Erreur transformation col {col} transfo {transformation.ordre}"
                )

                # Exclusion des lignes en erreur
                active_mask = active_mask & ~new_error_mask
                error_mask_global |= new_error_mask

        # Fin transformations colonne

    # SUPPRESSION LIGNES EN ERREUR
    df_clean = df_clean[~error_mask_global].copy()

    return df_clean, anomalies

### 2.5 Vérifier les valeurs NULL

In [ ]:
def handle_missing_values(
    df: pd.DataFrame,
    anomaly_df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Traite les valeurs manquantes dans df selon les règles de mappings :
    - Convertit les valeurs vides / null / NaN en pd.NA
    - Si nullable=True : laisse pd.NA
    - Si nullable=False et valeur_defaut définie : remplace les NA par la valeur par défaut
    - Sinon : les lignes avec NA deviennent anomalies
    """
    df_clean = df.copy()
    anomalies = anomaly_df.copy()

    for mapping in mappings:
        col = mapping.colonne_bdd
        if col not in df_clean.columns:
            continue

        # --- Étape 1 : standardiser les valeurs "vides" en pd.NA ---
        df_clean[col] = df_clean[col].replace(
            ["", " ", None, "null", "NULL", "None", "nan", "NAN", "na", "NA"], pd.NA
        )

        # --- Étape 2 : vérifier nullable ---
        mask_na = df_clean[col].isna()
        if not mask_na.any():
            # pas de valeur manquante → passer à la colonne suivante
            continue

        if mapping.nullable:
            # nullable = True → laisser pd.NA
            continue

        if mapping.valeur_defaut not in ["", " ", None, "null", "NULL", np.nan]:
            # nullable = False mais valeur par défaut définie → remplacer les NA
            df_clean[col] = df_clean[col].fillna(mapping.valeur_defaut)
        else:
            # nullable = False et pas de valeur par défaut → ce sont des anomalies
            # capturer les lignes en anomalies avant suppression
            rows_with_error = df_clean.loc[mask_na].copy()
            rows_with_error["erreur"] = f"La cellule {col} est null"
            anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)

            # Supprimer ces lignes du df_clean
            df_clean = df_clean.loc[~mask_na].copy()

    df_clean.reset_index(drop=True, inplace=True)
    anomalies.reset_index(drop=True, inplace=True)

    return df_clean, anomalies

### 2.6 Regarder si les colonnes sont convertibles dans le type cible

In [1227]:
import ast

def parse_array(x):
    if isinstance(x, list):
        return x

    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            return [parsed]
        except Exception:
            # cas "a;b;c"
            if ";" in x:
                return x.split(";")

    return None

def convert_column_type(
    df: pd.DataFrame,
    anomaly_df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> tuple[pd.DataFrame, pd.DataFrame]:

    df_clean = df.copy()
    anomalies = anomaly_df.copy()

    for mapping in mappings:
        col = mapping.colonne_bdd

        if col not in df_clean.columns:
            continue

        series = df_clean[col]

        #region --- Étape 1 : conversion selon type de toutes les lignes de la colonne ---
        if mapping.type_donnees == TypeDonnees.INT:
            # Conversion simple: numérique -> arrondi 0 décimale -> entier nullable
            converted = pd.to_numeric(series, errors='coerce').round(0).astype('Int64')
        elif mapping.type_donnees == TypeDonnees.DECIMAL:
            converted = pd.to_numeric(series, errors='coerce')
        elif mapping.type_donnees == TypeDonnees.STRING:
            converted = series.astype(str)
        elif mapping.type_donnees == TypeDonnees.ARRAY_DELIMITED_JSON:
            # Convertir les listes en string délimité par ";"
            separator = ";"
            converted = series.map(
                lambda x: separator.join(map(str, parse_array(x)))
                if parse_array(x) is not None else pd.NA
            )
        elif mapping.type_donnees == TypeDonnees.ARRAY_JSON:
            converted = series.map(
                lambda x: json.dumps(parse_array(x))
                if parse_array(x) is not None else pd.NA
            )
        elif mapping.type_donnees in [TypeDonnees.DATE, TypeDonnees.TIMESTAMP]:
            converted = pd.to_datetime(series, errors='coerce')
            if mapping.type_donnees == TypeDonnees.DATE:
                converted = converted.dt.date
        elif mapping.type_donnees == TypeDonnees.BOOLEAN:
            converted = series.map(lambda x: True if str(x).lower() in ['true','1']
                                   else False if str(x).lower() in ['false','0'] else pd.NA)
        else:
            converted = series
        #endregion

        #region --- Étape 2 : gérer les valeurs manquantes selon nullable / valeur par défaut ---
        mask_invalid = converted.isna()

        # Si valeur null ok alors ne rien faire de plus
        if mapping.nullable:
            # On laisse les NaN
            print("On laisse les NaN")
            df_clean[col] = converted
            continue

        if mapping.valeur_defaut not in [None, "", np.nan]:
            # Convertir valeur par défaut dans le bon type
            try:
                if mapping.type_donnees == TypeDonnees.INT:
                    # Même logique pour le défaut: numérique -> arrondi -> int
                    default_value = int(round(float(mapping.valeur_defaut)))
                elif mapping.type_donnees == TypeDonnees.DECIMAL:
                    default_value = pd.to_numeric(mapping.valeur_defaut)
                elif mapping.type_donnees in [TypeDonnees.STRING, TypeDonnees.ARRAY_DELIMITED_JSON, TypeDonnees.ARRAY_JSON]:
                    default_value = str(mapping.valeur_defaut)
                elif mapping.type_donnees in [TypeDonnees.DATE, TypeDonnees.TIMESTAMP]:
                    default_value = pd.to_datetime(mapping.valeur_defaut)
                    if mapping.type_donnees == TypeDonnees.DATE:
                        default_value = default_value.date()
                elif mapping.type_donnees == TypeDonnees.BOOLEAN:
                    default_value = True if str(mapping.valeur_defaut).lower() in ['true','1'] else False
                else:
                    default_value = mapping.valeur_defaut
            except Exception:
                # Mettre à jour les valeurs valides
                df_clean.loc[~mask_invalid, col] = converted.loc[~mask_invalid]
                # Valeur par défaut invalide : toutes les lignes invalides deviennent anomalies
                rows_with_error = df_clean.loc[mask_invalid].copy()
                rows_with_error["erreur"] = f"La cellule {col} n'est pas convertible et la valeur par défaut '{mapping.valeur_defaut}' est invalide"
                anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)
                # On supprime ces lignes
                df_clean = df_clean.loc[~mask_invalid]
                continue

            # Remplacer les NaN par valeur par défaut
            converted[mask_invalid] = default_value
            df_clean[col] = converted
        else:
            # Mettre les valeurs converties (IMPORTANT)
            df_clean[col] = converted

            # Identifier les lignes invalides
            rows_with_error = df_clean.loc[mask_invalid].copy()
            rows_with_error["erreur"] = f"La cellule {col} n'est pas convertible en {mapping.type_donnees.value}"

            # Ajouter aux anomalies
            anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)

            # Garder uniquement les lignes valides
            df_clean = df_clean.loc[~mask_invalid]
        #endregion

    df_clean.reset_index(drop=True, inplace=True)
    anomalies.reset_index(drop=True, inplace=True)

    return df_clean, anomalies


### 2.7 Exécuter les vérifications en fonction des contraintes des colonnes

In [1228]:
def _build_numeric_constraint_mask(
    series: pd.Series,
    constraint: NumericConstraint,
    index: pd.Index
) -> pd.Series:
    mask_invalid = pd.Series(False, index=index)
    numeric_series = pd.to_numeric(series, errors='coerce')

    if constraint.nb_min is not None:
        mask_invalid = mask_invalid | (numeric_series < constraint.nb_min)

    if constraint.nb_max is not None:
        mask_invalid = mask_invalid | (numeric_series > constraint.nb_max)

    if constraint.nb_decimal is not None:
        # Vérifie le nombre de décimales en analysant la valeur d'origine en string
        def too_many_decimals(value):
            if pd.isna(value):
                return False
            s = str(value).strip()
            if s == "" or s.lower() in ["nan", "none", "null"]:
                return False
            if "." in s:
                return len(s.split(".")[-1]) > constraint.nb_decimal
            return False

        decimal_mask = series.apply(too_many_decimals)
        mask_invalid = mask_invalid | decimal_mask

    return mask_invalid


def _build_string_constraint_mask(
    series: pd.Series,
    constraint: StringConstraint,
    index: pd.Index
) -> pd.Series:
    mask_invalid = pd.Series(False, index=index)
    str_len = series.astype(str).str.len()

    if constraint.min_length is not None:
        mask_invalid = mask_invalid | (str_len < constraint.min_length)

    if constraint.max_length is not None:
        mask_invalid = mask_invalid | (str_len > constraint.max_length)

    return mask_invalid


def _build_date_constraint_mask(
    series: pd.Series,
    constraint: DateConstraint,
    index: pd.Index
) -> pd.Series:
    mask_invalid = pd.Series(False, index=index)
    dt_series = pd.to_datetime(series, errors='coerce')

    if constraint.date_min is not None:
        mask_invalid = mask_invalid | (dt_series < pd.to_datetime(constraint.date_min))

    if constraint.date_max is not None:
        mask_invalid = mask_invalid | (dt_series > pd.to_datetime(constraint.date_max))

    return mask_invalid


def check_column_constraint(
    df: pd.DataFrame,
    anomaly_df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Vérifie les contraintes de colonnes selon le type de contrainte défini dans mapping.constraint.
    Les lignes en erreur sont déplacées dans anomalies avec un message explicite.
    """

    df_clean = df.copy()
    anomalies = anomaly_df.copy()

    for mapping in mappings:
        col = mapping.colonne_bdd
        if col not in df_clean.columns:
            continue

        constraint = mapping.constraint
        if constraint is None:
            continue

        # Série sur le DataFrame courant (qui peut changer après suppression de lignes)
        series = df_clean[col]

        # --- Contraintes NUMERIC (INT/DECIMAL) ---
        if isinstance(constraint, NumericConstraint):
            mask_invalid = _build_numeric_constraint_mask(series, constraint, df_clean.index)

        # --- Contraintes STRING / ARRAY stringifiées ---
        elif isinstance(constraint, StringConstraint):
            mask_invalid = _build_string_constraint_mask(series, constraint, df_clean.index)

        # --- Contraintes DATE/TIMESTAMP ---
        elif isinstance(constraint, DateConstraint):
            mask_invalid = _build_date_constraint_mask(series, constraint, df_clean.index)

        # BOOLEAN : pas de contrainte dédiée
        else:
            continue

        if mask_invalid.any():
            rows_with_error = df_clean.loc[mask_invalid].copy()
            rows_with_error["erreur"] = f"La cellule {col} ne respecte pas les contraintes de la colonne"
            anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)

            # Supprimer les lignes invalides
            df_clean = df_clean.loc[~mask_invalid].copy()

    df_clean.reset_index(drop=True, inplace=True)
    anomalies.reset_index(drop=True, inplace=True)

    return df_clean, anomalies

## Partie 3 : Enregistrement du dataframe

#### 3.1 : Fonction pour save un dataframe dans un fichier

In [1229]:
def save_dataframe_to_csv(df, folder_path, file_name):
    """
    Enregistre un DataFrame pandas en fichier CSV.

    :param df: DataFrame pandas
    :param folder_path: Chemin du dossier de destination
    :param file_name: Nom du fichier (sans extension ou avec .csv)
    """

    # S'assurer que le dossier existe
    os.makedirs(folder_path, exist_ok=True)

    # Ajouter .csv si pas présent
    if not file_name.endswith(".csv"):
        file_name += ".csv"

    # Construire le chemin complet
    full_path = os.path.join(folder_path, file_name)

    # Sauvegarde
    df.to_csv(full_path, index=False)

    return full_path

#### 3.2 : Fonction pour enregistrer en BDD les dataframe

In [1230]:
# ICI QUE POUR LE NOTEBOOK
import sys
import os

sys.path.append(os.path.abspath(".."))
# FIN ICI QUE POUR LE NOTEBOOK

from src.data_pipeline.database import engine
from sqlalchemy.exc import SQLAlchemyError

def ingest_cleaned_data(file_path: str, pipeline: PipelineETL):
    # 1. Charger le CSV traité
    df = pd.read_csv(file_path)

    # 2. Envoyer dans la BDD
    # 'name' doit correspondre au tablename du modèle (ex: "profil_sante")
    try:
        df.to_sql(
            name=pipeline.table_nom,
            con=engine,
            if_exists='append',
            index=False
        )
        print(f"Ingestion réussie : {len(df)} lignes ajoutées.")

    except SQLAlchemyError as e:
        print("Erreur SQL :", e)

    except Exception as e:
        print("Erreur inattendue :", e)

#### 3.3 : Fonction principale pour save les df

In [1231]:
def loader_pipeline(df, anomalies, pipeline: PipelineETL):
    # Normalisation du chemin
    normalized_folder = normalize_path(pipeline.dossier_clean_emplacement)

    timestamp = datetime.now().strftime("_%Y%m%d_%H%M%S")
    file_name = pipeline.nom_fichier_fixe + pipeline.nom_fichier_variable + timestamp

    path = save_dataframe_to_csv(df, normalized_folder, file_name)

    # TEST POUR LES ANOMALIES
    file_name = pipeline.nom_fichier_fixe + pipeline.nom_fichier_variable + "_anomalies" + timestamp
    path_anomalie = save_dataframe_to_csv(anomalies, normalized_folder, file_name)
    # FIN TEST DES ANOMALIES

    # Enregistre en BDD
    ingest_cleaned_data(path, pipeline)

    return path

### Test

In [ ]:
# Pipeline pour exercisedb_hobby
def get_pipeline_exercisedb_hobby():    
    # Nom
    col_nom = ETLColumnMapping(
        id_etl_column_mapping=1,
        colonne_bdd="nom",
        colonne_fichier="name",
        in_file=True,
        type_donnees=TypeDonnees.STRING,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=0, max_length=150),
        transformations=[
        ]
    )
    
    # type
    col_exerciseType = ETLColumnMapping(
        id_etl_column_mapping=1,
        colonne_bdd="type_exercice",
        colonne_fichier="exerciseType",
        in_file=True,
        type_donnees=TypeDonnees.STRING,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=0, max_length=100),
        transformations=[
        ]
    )

    # muscle_cible
    col_muscle_cible = ETLColumnMapping(
        id_etl_column_mapping=4,
        colonne_bdd="muscles_principaux",
        colonne_fichier="targetMuscles",
        in_file=True,
        type_donnees=TypeDonnees.ARRAY_DELIMITED_JSON,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=0, max_length=100),
        transformations=[
            ETLColumnTransformation(
                id_transformation=5,
                id_etl_column_mapping=3,
                ordre=1,
                skip_error=False,
                type_transformation=TypeTransformation.ARRAY_UNIQUE,
                condition_fail_behavior=ConditionFailBehavior.ERROR
            )
        ]
    )
    
    # muscle_cible
    col_muscles_secondaires = ETLColumnMapping(
        id_etl_column_mapping=4,
        colonne_bdd="muscles_secondaires",
        colonne_fichier="secondaryMuscles",
        in_file=True,
        type_donnees=TypeDonnees.ARRAY_DELIMITED_JSON,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=0, max_length=100),
        transformations=[
            ETLColumnTransformation(
                id_transformation=5,
                id_etl_column_mapping=3,
                ordre=1,
                skip_error=False,
                type_transformation=TypeTransformation.ARRAY_UNIQUE,
                condition_fail_behavior=ConditionFailBehavior.ERROR
            )
        ]
    )

    # equipement
    col_equipement = ETLColumnMapping(
        id_etl_column_mapping=4,
        colonne_bdd="equipement",
        colonne_fichier="equipments",
        in_file=True,
        type_donnees=TypeDonnees.ARRAY_DELIMITED_JSON,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=0, max_length=100),
        transformations=[
            ETLColumnTransformation(
                id_transformation=5,
                id_etl_column_mapping=3,
                ordre=1,
                skip_error=False,
                type_transformation=TypeTransformation.ARRAY_UNIQUE,
                condition_fail_behavior=ConditionFailBehavior.ERROR
            )
        ]
    )

    # -------------------------
    # PIPELINE
    # -------------------------
    pipeline = PipelineETL(
        1,
        "Import exercisedb_hobby",
        "exercice",
        "\\raw\\",
        "exercisedb_hobby",
        "",
        ExtensionFichier.JSON,
        "/clean",
        True,
        [col_nom, col_exerciseType, col_muscle_cible, col_equipement, col_muscles_secondaires]
    )

    return pipeline
    

In [1233]:
from IPython.display import display # pour notebook test

pipeline_rules = get_pipeline_exercisedb_hobby()
pipeline_column_mapping = pipeline_rules.colonnes

dfs_matched_files = get_df_matched_files(pipeline_rules)

for df in dfs_matched_files:
    df_clean = column_mapper(df, pipeline_column_mapping)
    anomalies = generate_anomaly_dataframe(pipeline_column_mapping)

    print("--------------- Check Fin -----------")
    display(df_clean)
    print("----- anomalies FIN -----")
    display(anomalies)

    # supprime les espaces inutiles des string
    df_clean = clean_txt(df_clean)

    # Transformer les colonnes
    df_clean, anomalies = apply_transformations(df_clean, anomalies, pipeline_column_mapping)

    # Verifier les valeurs null
    df_clean, anomalies = handle_missing_values(df_clean, anomalies, pipeline_column_mapping)

    # Regarder si les colonnes sont convertible dans le type cible
    df_clean, anomalies = convert_column_type(df_clean, anomalies, pipeline_column_mapping)

    # Vérifier les contraintes métier de colonnes
    df_clean, anomalies = check_column_constraint(df_clean, anomalies, pipeline_column_mapping)

    print("--------------- Check Fin -----------")
    display(df_clean)
    print("----- anomalies FIN -----")
    display(anomalies)

    path = loader_pipeline(df_clean, anomalies, pipeline_rules)
    
    print(f"Fichier clean enregistré ici : {path}")

--------------- Check Fin -----------


,nom,type_exercice,muscles_principaux,equipement,muscles_secondaires
0,es,STRENGTH,['OBLIQUES'],dzdqzdqzdzqd,"['1', '2', '1', '2', '3', '4', '5']"
1,10,STRENGTH,"['GLUTEUS MAXIMUS', 'QUADRICEPS']",['BODY WEIGHT'],"['ADDUCTOR MAGNUS', 'GLUTEUS MEDIUS', 'SOLEUS'..."
2,Sliding Floor Pulldown on Towel,STRENGTH,['LATISSIMUS DORSI'],"['BODY WEIGHT', 'ADDUCTOR BREVIS', 'TENSOR FAS...","['POSTERIOR DELTOID', 'TRICEPS BRACHII', 'TERE..."
3,Triceps Dip,STRENGTH,['TRICEPS BRACHII'],['BODY WEIGHT'],"['PECTORALIS MAJOR STERNAL HEAD', 'PECTORALIS ..."
4,Dumbbell Incline One Arm Hammer Press,STRENGTH,['RECTUS ABDOMINIS'],['DUMBBELL'],"['TERES MAJOR', 'PECTORALIS MAJOR CLAVICULAR H..."
5,Feet and Ankles Stretch,STRENGTH,"['SOLEUS', 'TIBIALIS ANTERIOR']",['BODY WEIGHT'],[]
6,Jump Squat,STRENGTH,"['GLUTEUS MAXIMUS', 'QUADRICEPS']",['BODY WEIGHT'],"['SOLEUS', 'ADDUCTOR MAGNUS']"
7,Dumbbell Single Leg Calf Raise,STRENGTH,['GASTROCNEMIUS'],['DUMBBELL'],['SOLEUS']
8,Peroneals Stretch,STRENGTH,['SOLEUS'],['BODY WEIGHT'],[]
9,Lying Double Legs Biceps Curl with Towel,STRENGTH,['BICEPS BRACHII'],['BODY WEIGHT'],"['BRACHIALIS', 'BRACHIORADIALIS']"


----- anomalies FIN -----


,nom,type_exercice,muscles_principaux,equipement,muscles_secondaires,erreur


--------------- Check Fin -----------


C:\Users\33631\AppData\Local\Temp\ipykernel_16792\2969678586.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df_clean.select_dtypes(include='object').columns
C:\Users\33631\AppData\Local\Temp\ipykernel_16792\2969678586.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/doc

,nom,type_exercice,muscles_principaux,equipement,muscles_secondaires
0,10,STRENGTH,GLUTEUS MAXIMUS;QUADRICEPS,BODY WEIGHT,ADDUCTOR MAGNUS;GLUTEUS MEDIUS;SOLEUS;TENSOR F...
1,Dumbbell Incline One Arm Hammer Press,STRENGTH,RECTUS ABDOMINIS,DUMBBELL,TERES MAJOR;PECTORALIS MAJOR CLAVICULAR HEAD;T...
2,Feet and Ankles Stretch,STRENGTH,SOLEUS;TIBIALIS ANTERIOR,BODY WEIGHT,
3,Jump Squat,STRENGTH,GLUTEUS MAXIMUS;QUADRICEPS,BODY WEIGHT,SOLEUS;ADDUCTOR MAGNUS
4,Dumbbell Single Leg Calf Raise,STRENGTH,GASTROCNEMIUS,DUMBBELL,SOLEUS
5,Peroneals Stretch,STRENGTH,SOLEUS,BODY WEIGHT,
6,Lying Double Legs Biceps Curl with Towel,STRENGTH,BICEPS BRACHII,BODY WEIGHT,BRACHIALIS;BRACHIORADIALIS
7,Calf Raise from Deficit with Chair Supported,STRENGTH,GASTROCNEMIUS,BODY WEIGHT,SOLEUS
8,Dumbbell Stiff Leg Deadlift,STRENGTH,ERECTOR SPINAE;GLUTEUS MAXIMUS,DUMBBELL,HAMSTRINGS
9,Single Leg Squat,STRENGTH,GLUTEUS MAXIMUS;QUADRICEPS,BODY WEIGHT,SOLEUS;ADDUCTOR MAGNUS


----- anomalies FIN -----


,nom,type_exercice,muscles_principaux,equipement,muscles_secondaires,erreur
0,es,STRENGTH,['OBLIQUES'],dzdqzdqzdzqd,"['1', '2', '1', '2', '3', '4', '5']",Erreur conversion type col equipement transfo 1
1,Seated Row with Towel,STRENGTH,INFRASPINATUS;TRAPEZIUS UPPER FIBERS;LATISSIMU...,BODY WEIGHT,POSTERIOR DELTOID;BRACHIORADIALIS;BRACHIALIS,La cellule muscles_principaux ne respecte pas ...
2,Chin-ups,STRENGTH,INFRASPINATUS;LATISSIMUS DORSI;TRAPEZIUS LOWER...,BODY WEIGHT,BRACHIORADIALIS;PECTORALIS MAJOR STERNAL HEAD;...,La cellule muscles_principaux ne respecte pas ...
3,Suspended Row,STRENGTH,INFRASPINATUS;TRAPEZIUS UPPER FIBERS;TRAPEZIUS...,BODY WEIGHT,PECTORALIS MAJOR STERNAL HEAD;BRACHIORADIALIS;...,La cellule muscles_principaux ne respecte pas ...
4,Elliptical Machine Walk,CARDIO,ANTERIOR DELTOID;HAMSTRINGS;POSTERIOR DELTOID;...,LEVERAGE MACHINE,,La cellule muscles_principaux ne respecte pas ...
5,Sliding Floor Pulldown on Towel,STRENGTH,LATISSIMUS DORSI,BODY WEIGHT;ADDUCTOR BREVIS;TENSOR FASCIAE LAT...,POSTERIOR DELTOID;TRICEPS BRACHII;TERES MINOR;...,La cellule muscles_secondaires ne respecte pas...
6,Triceps Dip,STRENGTH,TRICEPS BRACHII,BODY WEIGHT,PECTORALIS MAJOR STERNAL HEAD;PECTORALIS MAJOR...,La cellule muscles_secondaires ne respecte pas...


Ingestion réussie : 18 lignes ajoutées.
Fichier clean enregistré ici : ..\data\clean\exercisedb_hobby_20260407_115113.csv
